## Tool Biniding

Tool Binding is the step where you register tools with a Language Model (LLM) so that:

1. The LLM knows what tools are available.
2. It knows what each tool does(via description)
3. It knows what input format to use(via schema)

In [38]:
from langchain_openai import AzureChatOpenAI
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage, AIMessage
import requests

In [39]:
@tool
def multiply(a:int, b:int)->int:
    '''Given 2 numbers a and b this tool returns a*b'''
    return a*b

In [40]:
print(multiply.invoke({'a':3,'b':10}))

30


In [41]:
multiply.name, multiply.description, multiply.args

('multiply',
 'Given 2 numbers a and b this tool returns a*b',
 {'a': {'title': 'A', 'type': 'integer'},
  'b': {'title': 'B', 'type': 'integer'}})

In [42]:
from dotenv import load_dotenv
load_dotenv()
import os

In [43]:
model=AzureChatOpenAI(
    azure_endpoint=os.getenv('AZURE_OPENAI_ENDPOINT'),
    azure_deployment=os.getenv('AZURE_OPENAI_DEPLOYMENT'),
    model=os.getenv('AZURE_OPENAI_MODEL_NAME'),
    api_version=os.getenv('AZURE_OPENAI_API_VERSION'),

)

In [44]:
llm_with_tools= model.bind_tools([multiply])

## Tool Calling

Tool Calling is the process where the LLM (Language Model) decides, during a conversation or task, that it needs to use a specific tool(function)- and generates a structured output with:

* the name of the tool
* and the arguments to call it with

The LLM does not actually run the tools - it just suggests the tool and the input arguments. The actual excecution is handled by LangChain or you.

In [45]:
llm_with_tools.invoke('Hi How are you?').content

"Hello! I'm just a virtual assistant, so I don't have feelings, but I'm here and ready to help you with anything you need. How can I assist you today?"

In [46]:
query=HumanMessage('can you multiply 3 with 10')


In [47]:
messages=[query]

In [48]:
messages

[HumanMessage(content='can you multiply 3 with 10', additional_kwargs={}, response_metadata={})]

In [49]:
llm_with_tools.invoke('Can you multiply 3 with 10?')

AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'call_g33sa4Nx0nEn94HF0jaQV6xL', 'function': {'arguments': '{"a":3,"b":10}', 'name': 'multiply'}, 'type': 'function'}], 'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 18, 'prompt_tokens': 62, 'total_tokens': 80, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4.1-2025-04-14', 'system_fingerprint': 'fp_07e970ab25', 'id': 'chatcmpl-Bc4KOJAaCnwlZwXCl27jJN8d3mX9e', 'service_tier': None, 'prompt_filter_results': [{'prompt_index': 0, 'content_filter_results': {'hate': {'filtered': False, 'severity': 'safe'}, 'jailbreak': {'detected': False, 'filtered': False}, 'self_harm': {'filtered': False, 'severity': 'safe'}, 'sexual': {'filtered': False, 'severity': 'safe'}, 'violence': {'filtered': False, 'severity': 'safe'}}}], 'finish_rea

## when calling AI message has content='' empty but has additional_kwargs={'tool_calls':[...........]}
we can call the tool call

In [50]:
result=llm_with_tools.invoke('Can you multiply 3 with 10').tool_calls[0]
print(result)

{'name': 'multiply', 'args': {'a': 3, 'b': 10}, 'id': 'call_oE6UiC3j7nLj5CVmewDY28LP', 'type': 'tool_call'}


## we got the tool name, args, id and type. so we should call the tool ourself.

# Tool Execution

Tool Execution is the step where the actual Python function (tool) is run using the input arguments that the LLM suggested during tool calling.


In [51]:
multiply.invoke(result['args'])

30

In [52]:
multiply.invoke(result)

ToolMessage(content='30', name='multiply', tool_call_id='call_oE6UiC3j7nLj5CVmewDY28LP')

## all flow

In [53]:
query=HumanMessage('can you multiply 100*6')

In [54]:
messages=[query]
print(messages)

[HumanMessage(content='can you multiply 100*6', additional_kwargs={}, response_metadata={})]


In [58]:
result=llm_with_tools.invoke(messages)
messages.append(result)

In [59]:
messages

[HumanMessage(content='can you multiply 100*6', additional_kwargs={}, response_metadata={}),
 AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'call_Kvj5aBg3Khx9MYAvDAPDZqlV', 'function': {'arguments': '{"a":100,"b":6}', 'name': 'multiply'}, 'type': 'function'}], 'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 18, 'prompt_tokens': 60, 'total_tokens': 78, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4.1-2025-04-14', 'system_fingerprint': 'fp_07e970ab25', 'id': 'chatcmpl-Bc4nkP44FfL7pLegpQnpfcVzobz60', 'service_tier': None, 'prompt_filter_results': [{'prompt_index': 0, 'content_filter_results': {'hate': {'filtered': False, 'severity': 'safe'}, 'jailbreak': {'detected': False, 'filtered': False}, 'self_harm': {'filtered': False, 'severity': 'safe'}, 'sexual': {'filtered': 

In [62]:
tools_result=multiply.invoke(result.tool_calls[0])
tools_result

ToolMessage(content='600', name='multiply', tool_call_id='call_Kvj5aBg3Khx9MYAvDAPDZqlV')

In [63]:
messages.append(tools_result)

In [64]:
messages

[HumanMessage(content='can you multiply 100*6', additional_kwargs={}, response_metadata={}),
 AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'call_Kvj5aBg3Khx9MYAvDAPDZqlV', 'function': {'arguments': '{"a":100,"b":6}', 'name': 'multiply'}, 'type': 'function'}], 'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 18, 'prompt_tokens': 60, 'total_tokens': 78, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4.1-2025-04-14', 'system_fingerprint': 'fp_07e970ab25', 'id': 'chatcmpl-Bc4nkP44FfL7pLegpQnpfcVzobz60', 'service_tier': None, 'prompt_filter_results': [{'prompt_index': 0, 'content_filter_results': {'hate': {'filtered': False, 'severity': 'safe'}, 'jailbreak': {'detected': False, 'filtered': False}, 'self_harm': {'filtered': False, 'severity': 'safe'}, 'sexual': {'filtered': 

In [65]:
final=llm_with_tools.invoke(messages)

In [66]:
final

AIMessage(content='100 multiplied by 6 equals 600.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 11, 'prompt_tokens': 85, 'total_tokens': 96, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4.1-2025-04-14', 'system_fingerprint': 'fp_07e970ab25', 'id': 'chatcmpl-Bc4qro6a4X7XSNcax5TyGqeHLunRk', 'service_tier': None, 'prompt_filter_results': [{'prompt_index': 0, 'content_filter_results': {'hate': {'filtered': False, 'severity': 'safe'}, 'jailbreak': {'detected': False, 'filtered': False}, 'self_harm': {'filtered': False, 'severity': 'safe'}, 'sexual': {'filtered': False, 'severity': 'safe'}, 'violence': {'filtered': False, 'severity': 'safe'}}}], 'finish_reason': 'stop', 'logprobs': None, 'content_filter_results': {'hate': {'filtered': False, 'severity': 'safe'}, 'pr

In [67]:
final.content

'100 multiplied by 6 equals 600.'